# PROJETO EXPRO — EXplosion PROtection
**ER Dynamics Group** · Demonstração do sistema de bloqueio e alerta para detonações (NR-19), com **dados fictícios**.

1. Rode as células **1**, **2** e **3** em ordem (▶ de cada uma, ou *Ambiente de execução → Executar tudo*).
2. No painel, entre com um usuário: **Ana** (Técnica de Segurança) PIN `1234` · **Carlos** (Blaster) PIN `0000`.
3. Para mudar os dados fictícios, edite a célula 1 e rode tudo de novo.


In [ ]:
#@title 1. Dados fictícios (edite à vontade) { display-mode: "form" }
TERRENO = "rocha_dura"  #@param ["solo", "rocha_branda", "rocha_dura"]
ACELERAR_TEMPO = 1  #@param {type:"slider", min:1, max:5, step:1}
USAR_VOZ = True  #@param {type:"boolean"}

CONFIG = {
    "sistema": {
        "explosivo_xy": (0, 0),  # metros (x = leste, y = norte)
        "raios_terreno_m": {"solo": 150, "rocha_branda": 200, "rocha_dura": 250},
        "terreno": TERRENO,
        "duracao_nivel3_s": 15,
        "duracao_nivel2_s": 30,
        "intervalo_entre_avisos_s": 30,
        "contagem_regressiva_s": 30,
        "janela_disparo_s": 60,
        "timeout_sinal_s": 10,
        "bloquear_sem_sinal": True,
    },
    "usuarios": [
        {"login": "tecnico", "nome": "Ana", "perfil": "tecnico", "pin": "1234"},
        {"login": "blaster", "nome": "Carlos", "perfil": "operario", "pin": "0000"},
    ],
    # "Capacetes" com localizador: posição inicial de cada trabalhador (metros)
    "trabalhadores_simulados": [
        {"id": "CAP-01", "nome": "João", "x": 60, "y": 40},
        {"id": "CAP-02", "nome": "Maria", "x": -120, "y": 90},
        {"id": "CAP-03", "nome": "Pedro", "x": 150, "y": -170},
        {"id": "CAP-04", "nome": "Lucas", "x": 320, "y": 210},
        {"id": "CAP-05", "nome": "Rafaela", "x": -290, "y": -240},
    ],
    "simulacao": {
        "velocidade_evacuacao_m_s": 8,
        "acelerar_tempo": ACELERAR_TEMPO,
        "usar_voz": USAR_VOZ,
    },
}
print("Dados carregados.")


In [ ]:
#@title 2. Código do sistema (só rodar) { display-mode: "form" }
from __future__ import annotations

# ============================================================================
# expro/nucleo.py
# ============================================================================
import functools
import hashlib
import hmac
import math
import threading
import time
from dataclasses import dataclass, field, fields
from enum import Enum
from typing import Callable, Iterable, Optional, Protocol

# ---------------------------------------------------------------------------
# Usuários e perfis (Atores 1 e 2, RN-02)
# ---------------------------------------------------------------------------


class Perfil(str, Enum):
    TECNICO = "tecnico"  # Ator 1 — Técnico de Segurança
    OPERARIO = "operario"  # Ator 2 — Operário / Blaster


ROTULO_PERFIL = {Perfil.TECNICO: "Técnico de Segurança", Perfil.OPERARIO: "Operário / Blaster"}


@dataclass(frozen=True)
class Usuario:
    login: str
    nome: str
    perfil: Perfil
    pin_hash: str

    @staticmethod
    def gerar_hash(login: str, pin: str) -> str:
        return hashlib.sha256(f"expro|{login}|{pin}".encode("utf-8")).hexdigest()

    @classmethod
    def criar(cls, login: str, nome: str, perfil: Perfil | str, pin: str) -> Usuario:
        return cls(login, nome, Perfil(perfil), cls.gerar_hash(login, pin))

    def confere_pin(self, pin: str) -> bool:
        return hmac.compare_digest(self.pin_hash, self.gerar_hash(self.login, pin))

    @property
    def eh_tecnico(self) -> bool:
        return self.perfil is Perfil.TECNICO


def usuarios_de_config(lista: Iterable[dict]) -> list[Usuario]:
    """Aceita {"login", "nome", "perfil", "pin"} ou {"...", "pin_sha256"}."""
    usuarios = []
    for item in lista:
        if "pin_sha256" in item:
            usuarios.append(Usuario(item["login"], item["nome"], Perfil(item["perfil"]), item["pin_sha256"]))
        else:
            usuarios.append(Usuario.criar(item["login"], item["nome"], item["perfil"], str(item["pin"])))
    return usuarios


# ---------------------------------------------------------------------------
# Alertas sonoros (RF-03 — matriz de níveis)
# ---------------------------------------------------------------------------


@dataclass(frozen=True)
class Alerta:
    nivel: int
    titulo: str
    mensagem: str
    duracao_s: Optional[float]  # None = contínuo, até interrupção manual

    @property
    def continuo(self) -> bool:
        return self.duracao_s is None


def criar_alertas(duracao_nivel3_s: float = 15, duracao_nivel2_s: float = 30) -> dict[int, Alerta]:
    return {
        3: Alerta(3, "POSICIONAMENTO",
                  "Posicionamento de explosivos concluído. Favor evacuar a área de risco.", duracao_nivel3_s),
        2: Alerta(2, "DETONAÇÃO IMINENTE",
                  "Detonação iminente! Evacuação imediata obrigatória.", duracao_nivel2_s),
        1: Alerta(1, "EMERGÊNCIA / INSTABILIDADE",
                  "Alerta de emergência! Explosivos instáveis na área.", None),
    }


# ---------------------------------------------------------------------------
# Configuração
# ---------------------------------------------------------------------------


@dataclass
class Configuracao:
    explosivo_xy: tuple[float, float] = (0.0, 0.0)
    # Raio de risco por tipo de terreno, escolhido manualmente (ESC-04). Valores fictícios.
    raios_terreno_m: dict[str, float] = field(
        default_factory=lambda: {"solo": 150.0, "rocha_branda": 200.0, "rocha_dura": 250.0})
    terreno: str = "rocha_dura"
    duracao_nivel3_s: float = 15
    duracao_nivel2_s: float = 30
    intervalo_entre_avisos_s: float = 30  # cooldown (Ator 3 — temporizador)
    contagem_regressiva_s: float = 30
    janela_disparo_s: float = 60  # tempo em que a trava fica liberada aguardando o disparo
    timeout_sinal_s: float = 10  # capacete sem sinal há mais que isso = "sem sinal"
    bloquear_sem_sinal: bool = True  # fail-safe: capacete sem sinal conta como presente
    capacetes_cadastrados: tuple[str, ...] = ()

    @classmethod
    def de_dict(cls, dados: dict) -> Configuracao:
        nomes = {f.name for f in fields(cls)}
        desconhecidos = set(dados) - nomes
        if desconhecidos:
            raise ValueError(f"Parâmetros desconhecidos em [sistema]: {', '.join(sorted(desconhecidos))}")
        dados = dict(dados)
        if "explosivo_xy" in dados:
            dados["explosivo_xy"] = tuple(float(v) for v in dados["explosivo_xy"])
        if "raios_terreno_m" in dados:
            dados["raios_terreno_m"] = {k: float(v) for k, v in dados["raios_terreno_m"].items()}
        if "capacetes_cadastrados" in dados:
            dados["capacetes_cadastrados"] = tuple(dados["capacetes_cadastrados"])
        cfg = cls(**dados)
        cfg.validar()
        return cfg

    def validar(self) -> None:
        if self.terreno not in self.raios_terreno_m:
            raise ValueError(f"Terreno '{self.terreno}' não está em raios_terreno_m.")
        if any(r <= 0 for r in self.raios_terreno_m.values()):
            raise ValueError("Todos os raios de risco devem ser positivos.")
        for nome in ("duracao_nivel3_s", "duracao_nivel2_s", "contagem_regressiva_s",
                     "janela_disparo_s", "timeout_sinal_s"):
            if getattr(self, nome) <= 0:
                raise ValueError(f"{nome} deve ser positivo.")
        if self.intervalo_entre_avisos_s < 0:
            raise ValueError("intervalo_entre_avisos_s não pode ser negativo.")


# ---------------------------------------------------------------------------
# Interfaces dos adaptadores (o que o hardware precisa oferecer)
# ---------------------------------------------------------------------------


@dataclass(frozen=True)
class Leitura:
    """Posição de um capacete, em metros (x = leste, y = norte) no referencial do canteiro."""
    id: str
    x: float
    y: float
    instante: float
    nome: str = ""


class FonteLocalizacao(Protocol):
    def ler(self, agora: float) -> Iterable[Leitura]: ...


class SaidaAudio(Protocol):
    def tocar(self, alerta: Alerta) -> None: ...
    def parar(self) -> None: ...


class TravaDetonacao(Protocol):
    def liberar(self) -> None: ...
    def bloquear(self) -> None: ...


# ---------------------------------------------------------------------------
# Varredura da área (ESC-03)
# ---------------------------------------------------------------------------


class Situacao(str, Enum):
    DENTRO = "dentro do raio"
    FORA = "fora do raio"
    SEM_SINAL = "sem sinal"


@dataclass(frozen=True)
class Pessoa:
    id: str
    nome: str
    x: Optional[float]
    y: Optional[float]
    distancia_m: Optional[float]
    situacao: Situacao
    idade_leitura_s: Optional[float]


@dataclass(frozen=True)
class Varredura:
    raio_m: float
    pessoas: tuple[Pessoa, ...]

    @property
    def dentro(self) -> list[Pessoa]:
        return [p for p in self.pessoas if p.situacao is Situacao.DENTRO]

    @property
    def sem_sinal(self) -> list[Pessoa]:
        return [p for p in self.pessoas if p.situacao is Situacao.SEM_SINAL]


# ---------------------------------------------------------------------------
# Estados, eventos e resultados
# ---------------------------------------------------------------------------


class Estado(str, Enum):
    PREPARACAO = "PREPARAÇÃO"
    TESTE_AUDIO = "TESTE DE ÁUDIO"
    POSICIONAMENTO = "POSICIONAMENTO / EVACUAÇÃO"
    CONTAGEM = "CONTAGEM REGRESSIVA"
    DETONACAO_LIBERADA = "DETONAÇÃO LIBERADA"
    AGUARDANDO_REENTRADA = "AGUARDANDO REENTRADA"
    EMERGENCIA = "EMERGÊNCIA"


@dataclass(frozen=True)
class Evento:
    instante: float  # segundos desde o início do sistema
    tipo: str  # "info" | "aviso" | "bloqueio" | "emergencia" | "sucesso"
    texto: str

    def __str__(self) -> str:
        return f"[{self.instante:7.1f} s] {self.tipo.upper():<10} {self.texto}"


@dataclass(frozen=True)
class Resultado:
    ok: bool
    mensagem: str

    def __bool__(self) -> bool:
        return self.ok


@dataclass(frozen=True)
class Status:
    estado: Estado
    terreno: str
    raio_m: float
    pessoas_no_raio: int
    sem_sinal: int
    monitorados: int
    alerta: Optional[Alerta]
    alerta_restante_s: Optional[float]
    intervalo_restante_s: float
    contagem_restante_s: Optional[float]
    janela_restante_s: Optional[float]
    detonacao_liberada: bool


def _sincronizado(metodo):
    """Sensores, botões e interface podem chamar o sistema de threads diferentes."""
    @functools.wraps(metodo)
    def envoltorio(self, *args, **kwargs):
        with self._lock:
            return metodo(self, *args, **kwargs)
    return envoltorio


# ---------------------------------------------------------------------------
# Sistema EXPRO
# ---------------------------------------------------------------------------


class SistemaExpro:
    def __init__(self, config: Configuracao, localizador: FonteLocalizacao, audio: SaidaAudio,
                 trava: TravaDetonacao, usuarios: Iterable[Usuario] = (),
                 relogio: Callable[[], float] = time.monotonic):
        config.validar()
        self.config = config
        self.localizador = localizador
        self.audio = audio
        self.trava = trava
        self.usuarios = {u.login: u for u in usuarios}
        self.relogio = relogio
        self.alertas = criar_alertas(config.duracao_nivel3_s, config.duracao_nivel2_s)

        self.estado = Estado.PREPARACAO
        self.eventos: list[Evento] = []
        self.ouvintes: list[Callable[[Evento], None]] = []
        self.leituras: dict[str, Leitura] = {}
        self.alerta_ativo: Optional[Alerta] = None
        self.detonacao_liberada = False

        self._lock = threading.RLock()
        self._inicio = relogio()
        self._fim_alerta: Optional[float] = None
        self._fim_intervalo = -math.inf
        self._fim_contagem: Optional[float] = None
        self._fim_janela: Optional[float] = None
        self._fila_teste: list[int] = []
        self._duracao_teste = 0.0
        self._estado_antes_emergencia = Estado.PREPARACAO
        self._baixados: set[str] = set()
        self._resumo_area: Optional[tuple[tuple[str, ...], tuple[str, ...]]] = None
        self._falha_leitura = False
        self.varredura = Varredura(self.raio_m, ())

        self._bloquear_trava()  # fail-safe: sempre inicia com a detonação bloqueada
        self._registrar("info", f"Sistema iniciado. Terreno: {self.config.terreno} "
                                f"(raio {self.raio_m:g} m). Detonação bloqueada.")

    # ------------------------------------------------------------ consulta
    @property
    def raio_m(self) -> float:
        return self.config.raios_terreno_m[self.config.terreno]

    def bloqueantes(self) -> list[Pessoa]:
        """Pessoas que impedem a detonação (RN-01)."""
        v = self.varredura
        return v.dentro + (v.sem_sinal if self.config.bloquear_sem_sinal else [])

    @_sincronizado
    def status(self) -> Status:
        agora = self.relogio()

        def restante(fim: Optional[float]) -> Optional[float]:
            return None if fim is None else max(0.0, fim - agora)

        v = self.varredura
        return Status(self.estado, self.config.terreno, self.raio_m, len(v.dentro), len(v.sem_sinal),
                      len(v.pessoas), self.alerta_ativo, restante(self._fim_alerta),
                      max(0.0, self._fim_intervalo - agora), restante(self._fim_contagem),
                      restante(self._fim_janela), self.detonacao_liberada)

    # ------------------------------------------------- monitoramento contínuo
    @_sincronizado
    def atualizar(self) -> Varredura:
        """Chamado periodicamente (ex.: a cada 0,5 s): lê sensores e aplica as regras."""
        agora = self.relogio()
        try:
            for leitura in self.localizador.ler(agora):
                self.leituras[leitura.id] = leitura
                self._baixados.discard(leitura.id)
            if self._falha_leitura:
                self._falha_leitura = False
                self._registrar("info", "Leitura dos localizadores restabelecida.")
        except Exception as erro:  # sensor com defeito não pode derrubar o sistema
            if not self._falha_leitura:
                self._falha_leitura = True
                self._registrar("aviso", f"Falha na leitura dos localizadores: {erro}")
        self.varredura = self._varrer(agora)
        self._notificar_area()
        self._controlar_audio(agora)
        self._controlar_sequencia(agora)
        return self.varredura

    def _varrer(self, agora: float) -> Varredura:
        ex, ey = self.config.explosivo_xy
        raio = self.raio_m
        ids = (set(self.leituras) | set(self.config.capacetes_cadastrados)) - self._baixados
        pessoas = []
        for id_ in sorted(ids):
            leitura = self.leituras.get(id_)
            if leitura is None:  # cadastrado, mas nunca enviou posição
                pessoas.append(Pessoa(id_, "", None, None, None, Situacao.SEM_SINAL, None))
                continue
            distancia = math.hypot(leitura.x - ex, leitura.y - ey)
            idade = agora - leitura.instante
            if idade > self.config.timeout_sinal_s:
                situacao = Situacao.SEM_SINAL
            elif distancia <= raio:
                situacao = Situacao.DENTRO
            else:
                situacao = Situacao.FORA
            pessoas.append(Pessoa(id_, leitura.nome, leitura.x, leitura.y, distancia, situacao, idade))
        return Varredura(raio, tuple(pessoas))

    def _notificar_area(self) -> None:
        dentro = tuple(p.id for p in self.varredura.dentro)
        sem_sinal = tuple(p.id for p in self.varredura.sem_sinal)
        anterior = self._resumo_area
        if anterior == (dentro, sem_sinal):
            return
        self._resumo_area = (dentro, sem_sinal)
        if anterior is None or anterior[0] != dentro:
            if dentro:
                self._registrar("aviso", f"{len(dentro)} pessoa(s) no raio de risco: {', '.join(dentro)}.")
            elif sem_sinal and self.config.bloquear_sem_sinal:
                self._registrar("aviso", "Nenhuma pessoa localizada dentro do raio, mas há "
                                         f"{len(sem_sinal)} capacete(s) sem sinal.")
            else:
                self._registrar("sucesso", "Nenhuma pessoa detectada no raio de risco.")
        antes_sem_sinal = set(anterior[1]) if anterior else set()
        for id_ in sorted(set(sem_sinal) - antes_sem_sinal):
            consequencia = ("tratado como presente na área (fail-safe)" if self.config.bloquear_sem_sinal
                            else "ignorado (bloquear_sem_sinal = false)")
            self._registrar("aviso", f"{id_}: localizador sem sinal — {consequencia}.")
        for id_ in sorted(antes_sem_sinal - set(sem_sinal)):
            if id_ not in self._baixados:
                self._registrar("info", f"{id_}: sinal do localizador restabelecido.")

    def _controlar_audio(self, agora: float) -> None:
        if self.alerta_ativo is None or self._fim_alerta is None or agora < self._fim_alerta:
            return
        if self.estado is Estado.TESTE_AUDIO:
            self._audio_parar()
            self.alerta_ativo = self._fim_alerta = None
            if self._fila_teste:
                self._tocar_alerta(self._fila_teste.pop(0), agora, self._duracao_teste, teste=True)
            else:
                self.estado = Estado.PREPARACAO
                self._registrar("sucesso", "Teste de áudio concluído: Níveis 3, 2 e 1 reproduzidos.")
        else:
            self._parar_alerta(agora, "concluído")

    def _controlar_sequencia(self, agora: float) -> None:
        if self.estado in (Estado.CONTAGEM, Estado.DETONACAO_LIBERADA):
            bloqueantes = self.bloqueantes()
            if bloqueantes:  # RN-01 vale durante toda a contagem e a janela de disparo
                self._bloquear_trava()
                if self.estado is Estado.CONTAGEM:
                    self._parar_alerta(agora, "interrompido")
                self._fim_contagem = self._fim_janela = None
                self.estado = Estado.POSICIONAMENTO
                ids = ", ".join(p.id for p in bloqueantes)
                self._registrar("bloqueio", f"RN-01: presença detectada no raio ({ids}). "
                                            "Contagem cancelada e detonação BLOQUEADA.")
                return
        if self.estado is Estado.CONTAGEM and agora >= self._fim_contagem:
            self._fim_contagem = None
            self._fim_janela = agora + self.config.janela_disparo_s
            self.estado = Estado.DETONACAO_LIBERADA
            self._liberar_trava()
            self._registrar("sucesso", "Contagem concluída com a área livre. Detonação LIBERADA "
                                       f"por {self.config.janela_disparo_s:g} s.")
        elif self.estado is Estado.DETONACAO_LIBERADA and agora >= self._fim_janela:
            self._bloquear_trava()
            self._fim_janela = None
            self.estado = Estado.POSICIONAMENTO
            self._registrar("aviso", "Janela de disparo encerrada sem confirmação. Detonação bloqueada novamente.")

    # ------------------------------------------------------------- ações
    @_sincronizado
    def autenticar(self, login: str, pin: str) -> Optional[Usuario]:
        usuario = self.usuarios.get(login)
        if usuario is not None and usuario.confere_pin(pin):
            self._registrar("info", f"Login: {usuario.nome} ({ROTULO_PERFIL[usuario.perfil]}).")
            return usuario
        self._registrar("aviso", f"Falha de autenticação para '{login}'.")
        return None

    @_sincronizado
    def selecionar_terreno(self, usuario: Optional[Usuario], terreno: str) -> Resultado:
        """ESC-04: o terreno é escolhido manualmente pelo operador; define o raio de risco."""
        self.atualizar()
        if usuario is None:
            return self._negar("Faça login para alterar o terreno.")
        if self.estado is not Estado.PREPARACAO:
            return self._negar("O terreno só pode ser alterado na etapa de PREPARAÇÃO.")
        if terreno not in self.config.raios_terreno_m:
            return self._negar(f"Terreno desconhecido: {terreno}.")
        self.config.terreno = terreno
        self._registrar("info", f"{usuario.nome} selecionou o terreno '{terreno}' (raio {self.raio_m:g} m).")
        self.atualizar()
        return Resultado(True, f"Terreno: {terreno} — raio de risco {self.raio_m:g} m.")

    @_sincronizado
    def testar_audio(self, usuario: Optional[Usuario], duracao_por_nivel_s: float = 3) -> Resultado:
        """Critério de aceitação do RF-03: modo de teste cobrindo os Níveis 3, 2 e 1."""
        self.atualizar()
        agora = self.relogio()
        if usuario is None:
            return self._negar("Faça login para testar o áudio.")
        if self.estado is not Estado.PREPARACAO:
            return self._negar("O teste de áudio só pode ser feito na etapa de PREPARAÇÃO.")
        espera = self._motivo_espera(agora)
        if espera:
            return self._negar(f"Teste de áudio indisponível: {espera}.")
        self.estado = Estado.TESTE_AUDIO
        self._duracao_teste = duracao_por_nivel_s
        self._fila_teste = [2, 1]
        self._registrar("info", f"{usuario.nome} iniciou o teste de áudio (Níveis 3, 2 e 1).")
        self._tocar_alerta(3, agora, duracao_por_nivel_s, teste=True)
        return Resultado(True, "Teste de áudio iniciado.")

    @_sincronizado
    def acionar_nivel3(self, usuario: Optional[Usuario]) -> Resultado:
        """RF-02: verificação da área + aviso Nível 3 (posicionamento concluído)."""
        self.atualizar()
        agora = self.relogio()
        if usuario is None:
            return self._negar("Faça login para acionar o aviso Nível 3.")
        if self.estado not in (Estado.PREPARACAO, Estado.POSICIONAMENTO):
            return self._negar(f"Aviso Nível 3 indisponível no estado {self.estado.value}.")
        espera = self._motivo_espera(agora)
        if espera:
            return self._negar(f"Aviso Nível 3 indisponível: {espera}.")
        dentro = len(self.varredura.dentro)
        self.estado = Estado.POSICIONAMENTO
        self._registrar("info", f"{usuario.nome} verificou a área: {dentro} pessoa(s) no raio de {self.raio_m:g} m.")
        self._tocar_alerta(3, agora)
        return Resultado(True, f"Aviso Nível 3 acionado. {dentro} pessoa(s) no raio de risco.")

    @_sincronizado
    def autorizar_detonacao(self, usuario: Optional[Usuario]) -> Resultado:
        """RF-01: Técnico autoriza a detonação e aciona o Nível 2 (RN-01 e RN-02)."""
        self.atualizar()
        agora = self.relogio()
        if usuario is None or not usuario.eh_tecnico:
            quem = usuario.nome if usuario else "Usuário não autenticado"
            return self._negar(f"RN-02: {quem} não tem permissão. Só o Técnico de Segurança autenticado "
                               "libera a detonação e o aviso Nível 2.", "bloqueio")
        if self.estado is Estado.PREPARACAO:
            return self._negar("Acione antes o aviso Nível 3 (posicionamento e evacuação).")
        if self.estado is not Estado.POSICIONAMENTO:
            return self._negar(f"Autorização indisponível no estado {self.estado.value}.")
        bloqueantes = self.bloqueantes()
        if bloqueantes:
            ids = ", ".join(p.id for p in bloqueantes)
            return self._negar(f"RN-01: detonação BLOQUEADA — {len(bloqueantes)} pessoa(s) no raio de risco "
                               f"({ids}).", "bloqueio")
        espera = self._motivo_espera(agora)
        if espera:
            return self._negar(f"Autorização indisponível: {espera}.")
        self.estado = Estado.CONTAGEM
        self._fim_contagem = agora + self.config.contagem_regressiva_s
        self._registrar("info", f"Detonação autorizada por {usuario.nome}. "
                                f"Contagem regressiva de {self.config.contagem_regressiva_s:g} s.")
        self._tocar_alerta(2, agora)
        return Resultado(True, f"Detonação autorizada. Contagem de {self.config.contagem_regressiva_s:g} s iniciada.")

    @_sincronizado
    def registrar_detonacao(self, usuario: Optional[Usuario]) -> Resultado:
        """Confirma que o disparo ocorreu; a área fica interditada até a validação (RN-04)."""
        self.atualizar()
        if usuario is None:
            return self._negar("Faça login para registrar a detonação.")
        if self.estado is not Estado.DETONACAO_LIBERADA:
            return self._negar("A detonação não está liberada.")
        self._bloquear_trava()
        self._fim_janela = None
        self.estado = Estado.AGUARDANDO_REENTRADA
        self._registrar("info", f"Detonação realizada (registrada por {usuario.nome}). Área interditada "
                                "até a validação presencial do Técnico de Segurança.")
        return Resultado(True, "Detonação registrada. Aguardando validação de reentrada.")

    @_sincronizado
    def validar_reentrada(self, usuario: Optional[Usuario]) -> Resultado:
        """RN-04: só o Técnico de Segurança, após vistoria no local, libera a reentrada."""
        self.atualizar()
        if usuario is None or not usuario.eh_tecnico:
            return self._negar("RN-04: a reentrada só pode ser validada pelo Técnico de Segurança, "
                               "após vistoria presencial.", "bloqueio")
        if self.estado is not Estado.AGUARDANDO_REENTRADA:
            return self._negar("Não há detonação aguardando validação de reentrada.")
        self.estado = Estado.PREPARACAO
        self._registrar("sucesso", f"RN-04: reentrada validada presencialmente por {usuario.nome}.")
        return Resultado(True, "Reentrada validada. Sistema em PREPARAÇÃO.")

    @_sincronizado
    def acionar_emergencia(self, origem: str = "painel") -> Resultado:
        """RN-03: qualquer pessoa pode acionar; Nível 1 contínuo e interrupção da contagem."""
        agora = self.relogio()
        if self.estado is Estado.EMERGENCIA:
            return Resultado(True, "Emergência já está ativa.")
        self._estado_antes_emergencia = self.estado
        self._bloquear_trava()
        self._fim_contagem = self._fim_janela = None
        self._fila_teste = []
        self.estado = Estado.EMERGENCIA
        self._registrar("emergencia", f"RN-03: botão de emergência acionado ({origem}). "
                                      "Sequência interrompida e detonação bloqueada.")
        self._tocar_alerta(1, agora)  # ignora o intervalo entre avisos
        return Resultado(True, "EMERGÊNCIA acionada: alerta Nível 1 contínuo.")

    @_sincronizado
    def encerrar_emergencia(self, usuario: Optional[Usuario]) -> Resultado:
        """Interrupção manual do Nível 1 (perfil elevado)."""
        agora = self.relogio()
        if usuario is None or not usuario.eh_tecnico:
            return self._negar("Só o Técnico de Segurança pode encerrar a emergência.", "bloqueio")
        if self.estado is not Estado.EMERGENCIA:
            return self._negar("Não há emergência ativa.")
        self._parar_alerta(agora, "interrompido manualmente")
        voltar_para_reentrada = self._estado_antes_emergencia is Estado.AGUARDANDO_REENTRADA
        self.estado = Estado.AGUARDANDO_REENTRADA if voltar_para_reentrada else Estado.PREPARACAO
        self._registrar("info", f"Emergência encerrada por {usuario.nome}. Estado: {self.estado.value}.")
        return Resultado(True, f"Emergência encerrada. Estado: {self.estado.value}.")

    @_sincronizado
    def dar_baixa_capacete(self, usuario: Optional[Usuario], id_capacete: str) -> Resultado:
        """Retira do monitoramento um capacete devolvido (ex.: fim de turno)."""
        if usuario is None or not usuario.eh_tecnico:
            return self._negar("Só o Técnico de Segurança pode dar baixa em capacetes.", "bloqueio")
        self._baixados.add(id_capacete)
        self.leituras.pop(id_capacete, None)
        self._registrar("info", f"{usuario.nome} deu baixa no capacete {id_capacete}.")
        self.atualizar()
        return Resultado(True, f"Capacete {id_capacete} retirado do monitoramento.")

    # ------------------------------------------------------------ internos
    def _motivo_espera(self, agora: float) -> Optional[str]:
        if self.alerta_ativo is not None:
            if self._fim_alerta is None:
                return f"áudio Nível {self.alerta_ativo.nivel} em reprodução"
            return (f"aguarde o fim do áudio Nível {self.alerta_ativo.nivel} "
                    f"({math.ceil(self._fim_alerta - agora)} s)")
        if agora < self._fim_intervalo:
            return f"intervalo entre avisos ({math.ceil(self._fim_intervalo - agora)} s restantes)"
        return None

    def _tocar_alerta(self, nivel: int, agora: float, duracao: Optional[float] = None, teste: bool = False) -> None:
        alerta = self.alertas[nivel]
        if self.alerta_ativo is not None:
            self._audio_parar()
        duracao = alerta.duracao_s if duracao is None else duracao
        self.alerta_ativo = alerta
        self._fim_alerta = None if duracao is None else agora + duracao
        try:
            self.audio.tocar(alerta)
        except Exception as erro:
            self._registrar("aviso", f"Falha na saída de áudio: {erro}")
        quanto = "contínuo" if duracao is None else f"{duracao:g} s"
        prefixo = "TESTE — " if teste else ""
        self._registrar("emergencia" if nivel == 1 and not teste else "aviso",
                        f"{prefixo}Áudio Nível {nivel} ({alerta.titulo}, {quanto}): \"{alerta.mensagem}\"")

    def _parar_alerta(self, agora: float, motivo: str) -> None:
        if self.alerta_ativo is None:
            return
        nivel = self.alerta_ativo.nivel
        self._audio_parar()
        self.alerta_ativo = self._fim_alerta = None
        self._fim_intervalo = agora + self.config.intervalo_entre_avisos_s
        self._registrar("info", f"Áudio Nível {nivel} {motivo}. Intervalo de "
                                f"{self.config.intervalo_entre_avisos_s:g} s até o próximo aviso.")

    def _audio_parar(self) -> None:
        try:
            self.audio.parar()
        except Exception as erro:
            self._registrar("aviso", f"Falha ao parar o áudio: {erro}")

    def _liberar_trava(self) -> None:
        self.trava.liberar()
        self.detonacao_liberada = True

    def _bloquear_trava(self) -> None:
        self.detonacao_liberada = False
        try:
            self.trava.bloquear()
        except Exception as erro:
            self._registrar("emergencia", f"FALHA AO BLOQUEAR A TRAVA DE DETONAÇÃO: {erro}")

    def _negar(self, texto: str, tipo: str = "aviso") -> Resultado:
        self._registrar(tipo, texto)
        return Resultado(False, texto)

    def _registrar(self, tipo: str, texto: str) -> None:
        evento = Evento(self.relogio() - self._inicio, tipo, texto)
        self.eventos.append(evento)
        del self.eventos[:-500]
        for ouvinte in list(self.ouvintes):
            try:
                ouvinte(evento)
            except Exception:
                pass  # erro na interface não interrompe as regras de segurança

# ============================================================================
# expro/sons.py
# ============================================================================
import io
import math
import struct
import wave

# Cada padrão é uma lista de trechos (duração_s, frequência_inicial_hz, frequência_final_hz); 0 Hz = silêncio.
PADROES_SIRENE = {
    3: [(0.35, 880, 880), (0.05, 0, 0), (0.60, 660, 660), (1.50, 0, 0)],  # "ding-dong" calmo
    2: [(0.25, 960, 960), (0.25, 720, 720)] * 4 + [(0.40, 0, 0)],  # bipe alternado rápido
    1: [(0.80, 600, 1400), (0.80, 1400, 600)] * 2,  # sirene contínua subindo e descendo
}


def gerar_wav(nivel: int, taxa: int = 16000, volume: float = 0.8) -> bytes:
    """Retorna um WAV curto (mono, 16 bits) próprio para tocar em loop."""
    amostras: list[int] = []
    fase = 0.0
    rampa = max(1, int(0.008 * taxa))  # evita estalos no início e fim de cada bipe
    for duracao, f_ini, f_fim in PADROES_SIRENE[nivel]:
        n = int(duracao * taxa)
        for i in range(n):
            if f_ini == 0:
                amostras.append(0)
                continue
            freq = f_ini + (f_fim - f_ini) * i / n
            fase += 2 * math.pi * freq / taxa
            envelope = min(1.0, i / rampa, (n - i) / rampa)
            valor = volume * envelope * (0.8 * math.sin(fase) + 0.2 * math.sin(3 * fase))
            amostras.append(int(max(-1.0, min(1.0, valor)) * 32767))
    buffer = io.BytesIO()
    with wave.open(buffer, "wb") as arquivo:
        arquivo.setnchannels(1)
        arquivo.setsampwidth(2)
        arquivo.setframerate(taxa)
        arquivo.writeframes(struct.pack(f"<{len(amostras)}h", *amostras))
    return buffer.getvalue()

# ============================================================================
# expro/simulacao.py
# ============================================================================
import math
import time
from dataclasses import dataclass
from typing import Callable, Iterable, Optional



@dataclass
class TrabalhadorSimulado:
    id: str
    nome: str
    x: float
    y: float
    sinal: bool = True
    destino: Optional[tuple[float, float]] = None


class LocalizadorSimulado:
    """Faz o papel dos localizadores nos capacetes, com posições fictícias editáveis."""

    def __init__(self, trabalhadores: Iterable[dict], velocidade_m_s: float = 8.0):
        self.trabalhadores = {
            t["id"]: TrabalhadorSimulado(t["id"], t.get("nome", t["id"]), float(t["x"]), float(t["y"]))
            for t in trabalhadores
        }
        self.velocidade_m_s = velocidade_m_s
        self._ultimo: Optional[float] = None

    def ler(self, agora: float) -> list[Leitura]:
        dt = 0.0 if self._ultimo is None else max(0.0, agora - self._ultimo)
        self._ultimo = agora
        leituras = []
        for t in self.trabalhadores.values():
            self._andar(t, dt)
            if t.sinal:
                leituras.append(Leitura(t.id, t.x, t.y, agora, t.nome))
        return leituras

    def _andar(self, t: TrabalhadorSimulado, dt: float) -> None:
        if t.destino is None or dt <= 0:
            return
        dx, dy = t.destino[0] - t.x, t.destino[1] - t.y
        distancia = math.hypot(dx, dy)
        passo = self.velocidade_m_s * dt
        if distancia <= passo:
            t.x, t.y = t.destino
            t.destino = None
        else:
            t.x += dx / distancia * passo
            t.y += dy / distancia * passo

    # Comandos usados pela demonstração para movimentar as pessoas
    def caminhar_para(self, id_: str, x: float, y: float) -> None:
        self.trabalhadores[id_].destino = (x, y)

    def _ponto_na_direcao(self, t: TrabalhadorSimulado, centro: tuple[float, float], distancia: float):
        angulo = math.atan2(t.y - centro[1], t.x - centro[0]) if (t.x, t.y) != centro else 0.0
        return centro[0] + distancia * math.cos(angulo), centro[1] + distancia * math.sin(angulo)

    def sair_do_raio(self, id_: str, centro: tuple[float, float], raio: float, folga: float = 40.0) -> None:
        t = self.trabalhadores[id_]
        if math.hypot(t.x - centro[0], t.y - centro[1]) < raio + folga:
            t.destino = self._ponto_na_direcao(t, centro, raio + folga)

    def entrar_no_raio(self, id_: str, centro: tuple[float, float], raio: float) -> None:
        t = self.trabalhadores[id_]
        t.destino = self._ponto_na_direcao(t, centro, raio * 0.6)

    def evacuar(self, centro: tuple[float, float], raio: float, folga: float = 40.0) -> None:
        for t in self.trabalhadores.values():
            if math.hypot(t.x - centro[0], t.y - centro[1]) <= raio:
                self.sair_do_raio(t.id, centro, raio, folga)

    def definir_sinal(self, id_: str, ligado: bool) -> None:
        self.trabalhadores[id_].sinal = ligado


class AudioConsole:
    """Alto-falantes fictícios: guarda (e opcionalmente imprime) o que seria tocado."""

    def __init__(self, imprimir: Optional[Callable[[str], None]] = None):
        self.imprimir = imprimir
        self.tocando: Optional[Alerta] = None
        self.historico: list[int] = []

    def tocar(self, alerta: Alerta) -> None:
        self.tocando = alerta
        self.historico.append(alerta.nivel)
        if self.imprimir:
            self.imprimir(f"   >> ALTO-FALANTES (Nível {alerta.nivel}): \"{alerta.mensagem}\"")

    def parar(self) -> None:
        if self.tocando and self.imprimir:
            self.imprimir(f"   >> ALTO-FALANTES: fim do Nível {self.tocando.nivel}")
        self.tocando = None


class TravaSimulada:
    """Trava fictícia do circuito de disparo."""

    def __init__(self):
        self.liberada = False

    def liberar(self) -> None:
        self.liberada = True

    def bloquear(self) -> None:
        self.liberada = False


class RelogioManual:
    """Relógio controlado pelo programa (simula o tempo sem esperar de verdade)."""

    def __init__(self, inicio: float = 0.0):
        self.t = inicio

    def __call__(self) -> float:
        return self.t


def montar_sistema_simulado(config: dict, relogio: Callable[[], float] = time.monotonic,
                            audio=None) -> tuple[SistemaExpro, LocalizadorSimulado]:
    cfg = Configuracao.de_dict(config.get("sistema", {}))
    velocidade = config.get("simulacao", {}).get("velocidade_evacuacao_m_s", 8.0)
    localizador = LocalizadorSimulado(config.get("trabalhadores_simulados", []), velocidade)
    usuarios = usuarios_de_config(config.get("usuarios", []))
    sistema = SistemaExpro(cfg, localizador, audio or AudioConsole(), TravaSimulada(), usuarios, relogio)
    return sistema, localizador


def demonstracao(config: dict, imprimir: Callable[[str], None] = print) -> SistemaExpro:
    """Roteiro automático com os dados fictícios: percorre RF-01..03 e RN-01..04."""
    relogio = RelogioManual()
    sistema, loc = montar_sistema_simulado(config, relogio, AudioConsole(imprimir))
    sistema.ouvintes.append(lambda evento: imprimir(str(evento)))
    tecnico = next(u for u in sistema.usuarios.values() if u.eh_tecnico)
    blaster = next(u for u in sistema.usuarios.values() if not u.eh_tecnico)
    centro = sistema.config.explosivo_xy
    cfg = sistema.config

    def esperar(segundos: float, passo: float = 0.5) -> None:
        fim = relogio.t + segundos
        while relogio.t < fim - 1e-9:
            relogio.t = min(fim, relogio.t + passo)
            sistema.atualizar()

    def etapa(titulo: str) -> None:
        s = sistema.status()
        imprimir(f"\n=== {titulo} ===\n    estado: {s.estado.value} | raio {s.raio_m:g} m | "
                 f"{s.pessoas_no_raio} pessoa(s) no raio | trava: "
                 f"{'LIBERADA' if s.detonacao_liberada else 'BLOQUEADA'}")

    def acao(usuario: Optional[Usuario], descricao: str, executar: Callable) -> None:
        imprimir(f"  > {usuario.nome if usuario else 'Alguém no canteiro'} tenta: {descricao}")
        imprimir(f"    resultado: {'OK' if executar().ok else 'NEGADO'}")

    sistema.atualizar()
    etapa("1. Explosivos posicionados: verificação da área")
    acao(blaster, "autorizar a detonação", lambda: sistema.autorizar_detonacao(blaster))  # RN-02
    acao(tecnico, "autorizar a detonação", lambda: sistema.autorizar_detonacao(tecnico))  # sem aviso Nível 3
    acao(blaster, "verificar a área e acionar o Nível 3", lambda: sistema.acionar_nivel3(blaster))  # RF-02
    loc.evacuar(centro, sistema.raio_m)

    etapa("2. Evacuação em andamento")
    esperar(5)
    acao(tecnico, "autorizar a detonação", lambda: sistema.autorizar_detonacao(tecnico))  # RN-01
    esperar(cfg.duracao_nivel3_s + cfg.intervalo_entre_avisos_s)

    etapa("3. Área livre: Técnico autoriza (Nível 2 + contagem)")
    acao(tecnico, "autorizar a detonação", lambda: sistema.autorizar_detonacao(tecnico))  # RF-01
    esperar(min(10, cfg.contagem_regressiva_s / 2))

    intruso = next(iter(loc.trabalhadores))
    etapa(f"4. {intruso} volta para o raio durante a contagem")
    loc.entrar_no_raio(intruso, centro, sistema.raio_m)
    for _ in range(int(60 / 0.5)):
        esperar(0.5)
        if sistema.estado is not Estado.CONTAGEM:
            break
    loc.sair_do_raio(intruso, centro, sistema.raio_m)
    esperar(cfg.intervalo_entre_avisos_s + 5)

    etapa("5. Nova autorização com a área livre")
    acao(tecnico, "autorizar a detonação", lambda: sistema.autorizar_detonacao(tecnico))
    esperar(cfg.contagem_regressiva_s + 1)
    acao(blaster, "registrar que a detonação ocorreu", lambda: sistema.registrar_detonacao(blaster))

    etapa("6. Pós-detonação: reentrada (RN-04)")
    acao(blaster, "liberar a reentrada", lambda: sistema.validar_reentrada(blaster))
    acao(tecnico, "liberar a reentrada após vistoria", lambda: sistema.validar_reentrada(tecnico))

    etapa("7. Botão de emergência (RN-03)")
    esperar(cfg.intervalo_entre_avisos_s)
    acao(blaster, "acionar o Nível 3", lambda: sistema.acionar_nivel3(blaster))
    esperar(3)
    acao(None, "apertar o botão de emergência", lambda: sistema.acionar_emergencia("botão físico"))
    esperar(20)
    acao(blaster, "encerrar a emergência", lambda: sistema.encerrar_emergencia(blaster))
    acao(tecnico, "encerrar a emergência", lambda: sistema.encerrar_emergencia(tecnico))

    etapa("Fim da demonstração")
    imprimir(f"    níveis de áudio reproduzidos, em ordem: {sistema.audio.historico}")
    return sistema

# ============================================================================
# expro/painel_jupyter.py
# ============================================================================
import base64
import html
import io
import math
import threading
import time
from typing import Callable, Optional

import ipywidgets as w
from IPython.display import HTML, display


COR_NIVEL = {3: "#1565c0", 2: "#e07b00", 1: "#c62828"}
COR_ESTADO = {
    Estado.PREPARACAO: "#455a64", Estado.TESTE_AUDIO: "#6a1b9a", Estado.POSICIONAMENTO: "#1565c0",
    Estado.CONTAGEM: "#e07b00", Estado.DETONACAO_LIBERADA: "#2e7d32",
    Estado.AGUARDANDO_REENTRADA: "#6d4c41", Estado.EMERGENCIA: "#c62828",
}
COR_EVENTO = {"info": "#455a64", "aviso": "#a15c00", "bloqueio": "#c62828",
              "emergencia": "#c62828", "sucesso": "#2e7d32"}
VERMELHO, VERDE, CINZA = "#c62828", "#2e7d32", "#9e9e9e"


class AudioNavegador:
    """Toca os alertas no navegador: sirene do nível em loop, intercalada com voz (gTTS) se disponível."""

    def __init__(self, saida: w.Output, usar_voz: bool = False):
        self.saida = saida
        self._sirene = {n: base64.b64encode(gerar_wav(n)).decode() for n in (1, 2, 3)}
        self._voz: dict[int, str] = self._gerar_vozes() if usar_voz else {}
        self._contador = 0

    @staticmethod
    def _gerar_vozes() -> dict[int, str]:
        try:
            from gtts import gTTS
            vozes = {}
            for alerta in criar_alertas().values():
                buffer = io.BytesIO()
                gTTS(alerta.mensagem, lang="pt", tld="com.br").write_to_fp(buffer)
                vozes[alerta.nivel] = base64.b64encode(buffer.getvalue()).decode()
            return vozes
        except Exception:  # sem internet ou sem gTTS: fica só a sirene
            return {}

    def tocar(self, alerta: Alerta) -> None:
        self._contador += 1
        sirene = f"data:audio/wav;base64,{self._sirene[alerta.nivel]}"
        voz = self._voz.get(alerta.nivel)
        if voz:
            v, s = f"expro-voz-{self._contador}", f"expro-sirene-{self._contador}"
            codigo = (f'<audio id="{v}" autoplay src="data:audio/mpeg;base64,{voz}" '
                      f'onended="document.getElementById(\'{s}\').play()"></audio>'
                      f'<audio id="{s}" src="{sirene}" '
                      f'onended="document.getElementById(\'{v}\').play()"></audio>')
        else:
            codigo = f'<audio autoplay loop src="{sirene}"></audio>'
        self.saida.outputs = ()
        self.saida.append_display_data(HTML(codigo))

    def parar(self) -> None:
        self.saida.outputs = ()


# ---------------------------------------------------------------------------
# Desenho (HTML/SVG)
# ---------------------------------------------------------------------------


def _estrela(cx: float, cy: float, externo: float, interno: float, pontas: int = 8) -> str:
    pontos = []
    for i in range(pontas * 2):
        r = externo if i % 2 == 0 else interno
        a = math.pi * i / pontas - math.pi / 2
        pontos.append(f"{cx + r * math.cos(a):.1f},{cy + r * math.sin(a):.1f}")
    return " ".join(pontos)


def svg_mapa(sistema: SistemaExpro, tamanho: int = 430) -> str:
    v = sistema.varredura
    cx, cy = sistema.config.explosivo_xy
    raio = v.raio_m
    alcance = raio * 1.3
    for p in v.pessoas:
        if p.x is not None:
            alcance = max(alcance, abs(p.x - cx) + 30, abs(p.y - cy) + 30)
    meio = tamanho / 2
    esc = (meio - 12) / alcance

    def px(x: float, y: float) -> tuple[float, float]:
        return meio + (x - cx) * esc, meio - (y - cy) * esc

    cor = VERMELHO if sistema.bloqueantes() else VERDE
    s = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {tamanho} {tamanho + 34}" '
         f'width="{tamanho}" height="{tamanho + 34}" style="background:#f3efe6;border-radius:12px;'
         f'font-family:Inter,Arial,sans-serif">']
    passo = 100 if alcance <= 700 else 250
    for r in range(passo, int(alcance) + 1, passo):
        s.append(f'<circle cx="{meio}" cy="{meio}" r="{r * esc:.1f}" fill="none" stroke="#d5ccb9" '
                 'stroke-dasharray="3 5"/>')
        s.append(f'<text x="{meio + 4}" y="{meio - r * esc + 12:.1f}" font-size="10" fill="#9a8f78">{r} m</text>')
    s.append(f'<circle cx="{meio}" cy="{meio}" r="{raio * esc:.1f}" fill="{cor}" fill-opacity="0.13" '
             f'stroke="{cor}" stroke-width="2.5"/>')
    s.append(f'<text x="{meio}" y="{meio + raio * esc - 8:.1f}" font-size="12" font-weight="700" '
             f'text-anchor="middle" fill="{cor}">raio de risco {raio:g} m</text>')
    s.append(f'<polygon points="{_estrela(meio, meio, 13, 6)}" fill="#ff8f00" stroke="#4e342e" stroke-width="1.5"/>')
    for p in v.pessoas:
        if p.x is None:
            continue
        x, y = px(p.x, p.y)
        preenchimento = {Situacao.DENTRO: VERMELHO, Situacao.FORA: VERDE}.get(p.situacao, CINZA)
        traco = ' stroke-dasharray="3 2"' if p.situacao is Situacao.SEM_SINAL else ""
        s.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="8" fill="{preenchimento}" stroke="#fff" '
                 f'stroke-width="2"{traco}/>')
        s.append(f'<text x="{x:.1f}" y="{y - 12:.1f}" font-size="11" font-weight="700" text-anchor="middle" '
                 f'fill="#263238">{html.escape(p.id)}</text>')
    legenda = [(VERMELHO, "no raio"), (VERDE, "fora do raio"), (CINZA, "sem sinal"), ("#ff8f00", "explosivo")]
    for i, (c, texto) in enumerate(legenda):
        lx = 14 + i * (tamanho - 20) / 4
        s.append(f'<circle cx="{lx + 6}" cy="{tamanho + 17}" r="6" fill="{c}"/>'
                 f'<text x="{lx + 16}" y="{tamanho + 21}" font-size="11" fill="#37474f">{texto}</text>')
    s.append("</svg>")
    return "".join(s)


def _caixa(conteudo: str, fundo: str, cor: str = "#fff") -> str:
    return (f'<div style="background:{fundo};color:{cor};padding:10px 12px;border-radius:10px;'
            f'margin-bottom:8px">{conteudo}</div>')


def html_status(sistema: SistemaExpro, usuario: Optional[Usuario]) -> str:
    st = sistema.status()
    partes = [_caixa(f'<div style="font-size:11px;opacity:.85">ESTADO</div>'
                     f'<div style="font-size:19px;font-weight:800">{st.estado.value}</div>',
                     COR_ESTADO[st.estado])]

    cor_pessoas = VERMELHO if st.pessoas_no_raio else VERDE
    sem_sinal = (f'<div style="color:{VERMELHO};font-weight:700">+ {st.sem_sinal} capacete(s) sem sinal</div>'
                 if st.sem_sinal else "")
    partes.append(_caixa(
        f'<div style="font-size:11px;color:#546e7a">PESSOAS NO RAIO ({st.terreno.replace("_", " ")}, '
        f'{st.raio_m:g} m)</div><div style="font-size:40px;font-weight:800;color:{cor_pessoas};'
        f'line-height:1.1">{st.pessoas_no_raio}</div>{sem_sinal}'
        f'<div style="font-size:11px;color:#78909c">{st.monitorados} capacete(s) monitorado(s)</div>',
        "#eceff1", "#263238"))

    if st.detonacao_liberada:
        trava = f"🔓 DETONAÇÃO LIBERADA · {math.ceil(st.janela_restante_s or 0)} s"
        partes.append(_caixa(f'<b style="font-size:16px">{trava}</b>', VERDE))
    else:
        partes.append(_caixa('<b style="font-size:16px">🔒 DETONAÇÃO BLOQUEADA</b>', "#263238"))

    if st.contagem_restante_s is not None:
        partes.append(_caixa(f'<div style="font-size:11px">CONTAGEM REGRESSIVA</div>'
                             f'<div style="font-size:34px;font-weight:800">{math.ceil(st.contagem_restante_s)} s</div>',
                             COR_NIVEL[2]))

    if st.alerta is not None:
        quanto = "contínuo" if st.alerta_restante_s is None else f"{math.ceil(st.alerta_restante_s)} s"
        partes.append(_caixa(f'<b>🔊 NÍVEL {st.alerta.nivel} — {st.alerta.titulo}</b> · {quanto}'
                             f'<div style="font-style:italic;margin-top:4px">"{html.escape(st.alerta.mensagem)}"</div>',
                             COR_NIVEL[st.alerta.nivel]))
    else:
        espera = (f" · próximo aviso em {math.ceil(st.intervalo_restante_s)} s"
                  if st.intervalo_restante_s > 0 else "")
        partes.append(_caixa(f"🔈 Sem alerta sonoro{espera}", "#eceff1", "#455a64"))

    quem = (f"{html.escape(usuario.nome)} ({ROTULO_PERFIL[usuario.perfil]})" if usuario else "ninguém")
    partes.append(f'<div style="font-size:12px;color:#546e7a">Usuário conectado: <b>{quem}</b></div>')
    return f'<div style="font-family:Inter,Arial,sans-serif;width:340px">{"".join(partes)}</div>'


def html_registro(sistema: SistemaExpro, linhas: int = 12) -> str:
    itens = []
    for e in reversed(sistema.eventos[-linhas:]):
        itens.append(f'<div style="border-left:4px solid {COR_EVENTO.get(e.tipo, "#455a64")};padding:2px 8px;'
                     f'margin:3px 0;background:#fafafa"><span style="color:#90a4ae">{e.instante:6.1f} s</span> '
                     f'{html.escape(e.texto)}</div>')
    return (f'<div style="font-family:Inter,Arial,sans-serif;font-size:12.5px;color:#263238">'
            f'<b>Registro de eventos</b>{"".join(itens)}</div>')


# ---------------------------------------------------------------------------
# Painel
# ---------------------------------------------------------------------------


class PainelExpro:
    _ativo: Optional[PainelExpro] = None

    def __init__(self, config: dict):
        if PainelExpro._ativo is not None:  # rodar a célula de novo substitui o painel anterior
            PainelExpro._ativo.desligar()
        PainelExpro._ativo = self

        sim = config.get("simulacao", {})
        self.acelerar = max(1.0, float(sim.get("acelerar_tempo", 1)))
        t0 = time.monotonic()
        self.saida_audio = w.Output()
        self.audio = AudioNavegador(self.saida_audio, bool(sim.get("usar_voz", False)))
        self.sistema, self.loc = montar_sistema_simulado(
            config, lambda: (time.monotonic() - t0) * self.acelerar, self.audio)
        self.usuario: Optional[Usuario] = None
        self._lock = threading.RLock()
        self._parar = threading.Event()
        self._montar_widgets()
        self._executar(None)

    # ------------------------------------------------------------ widgets
    def _botao(self, texto: str, estilo: str, acao: Callable[[], Optional[Resultado]],
               icone: str = "") -> w.Button:
        b = w.Button(description=texto, button_style=estilo, icon=icone, layout=w.Layout(width="auto"))
        b.on_click(lambda _: self._executar(acao))
        return b

    def _montar_widgets(self) -> None:
        s = self.sistema
        self.mapa = w.HTML()
        self.status = w.HTML()
        self.registro = w.HTML()
        self.mensagem = w.HTML()

        self.sel_usuario = w.Dropdown(
            options=[(f"{u.nome} — {ROTULO_PERFIL[u.perfil]}", u.login) for u in s.usuarios.values()],
            layout=w.Layout(width="320px"))
        self.pin = w.Password(placeholder="PIN", layout=w.Layout(width="90px"))
        self.sel_terreno = w.Dropdown(options=list(s.config.raios_terreno_m), value=s.config.terreno,
                                      layout=w.Layout(width="150px"))
        self.sel_pessoa = w.Dropdown(options=[(f"{t.id} ({t.nome})", t.id) for t in self.loc.trabalhadores.values()],
                                     layout=w.Layout(width="190px"))
        self.evacuacao_auto = w.Checkbox(value=True, indent=False,
                                         description="pessoas evacuam sozinhas ao ouvir o Nível 3")
        self.relogio = w.Play(interval=500, min=0, max=10 ** 9, step=1, layout=w.Layout(width="auto"))
        self.relogio.observe(lambda _: self._executar(None), names="value")

        titulo = lambda t: w.HTML(f'<b style="font-family:Inter,Arial,sans-serif;color:#37474f">{t}</b>')
        acesso = w.HBox([self.sel_usuario, self.pin, self._botao("Entrar", "", self._entrar, "sign-in"),
                         self._botao("Sair", "", self._sair, "sign-out")], layout=w.Layout(flex_flow="row wrap"))
        operacao = w.HBox([
            self._botao("Nível 3 · verificar área", "info", self._nivel3, "bullhorn"),
            self._botao("Nível 2 · autorizar detonação", "warning", lambda: s.autorizar_detonacao(self.usuario), "key"),
            self._botao("Registrar detonação", "", lambda: s.registrar_detonacao(self.usuario), "bolt"),
            self._botao("Validar reentrada", "success", lambda: s.validar_reentrada(self.usuario), "check"),
        ], layout=w.Layout(flex_flow="row wrap"))
        seguranca = w.HBox([
            self._botao("EMERGÊNCIA · Nível 1", "danger", lambda: s.acionar_emergencia("botão do painel"),
                        "exclamation-triangle"),
            self._botao("Encerrar emergência", "", lambda: s.encerrar_emergencia(self.usuario), "stop"),
            self._botao("Testar áudio 3·2·1", "", lambda: s.testar_audio(self.usuario), "volume-up"),
            self.sel_terreno,
            self._botao("Aplicar terreno", "", lambda: s.selecionar_terreno(self.usuario, self.sel_terreno.value)),
        ], layout=w.Layout(flex_flow="row wrap"))
        simulacao = w.HBox([
            self.sel_pessoa,
            self._botao("Entrar no raio", "", lambda: self._mover(True), "arrow-right"),
            self._botao("Sair do raio", "", lambda: self._mover(False), "arrow-left"),
            self._botao("Liga/desliga sinal", "", self._alternar_sinal, "signal"),
            self._botao("Evacuar todos", "", self._evacuar, "users"),
        ], layout=w.Layout(flex_flow="row wrap"))

        self.interface = w.VBox([
            w.HTML('<div style="font-family:Inter,Arial,sans-serif;background:#263238;color:#ffc107;'
                   'padding:10px 14px;border-radius:10px;font-weight:800;font-size:18px">PROJETO EXPRO '
                   '<span style="color:#fff;font-weight:500;font-size:13px">· EXplosion PROtection · '
                   'ER Dynamics Group · demonstração com dados fictícios</span></div>'),
            w.HBox([self.mapa, self.status], layout=w.Layout(flex_flow="row wrap", grid_gap="14px")),
            self.mensagem,
            titulo("Acesso"), acesso,
            titulo("Operação"), operacao, seguranca,
            titulo("Simulação dos capacetes (dados fictícios)"), simulacao, self.evacuacao_auto,
            w.HBox([titulo("Relógio:"), self.relogio,
                    w.HTML('<span style="font-size:12px;color:#78909c">atualiza sozinho; se o painel '
                           'parecer parado, clique ▶</span>')]),
            self.registro,
            self.saida_audio,
        ])

    # ------------------------------------------------------------ ações
    def _entrar(self) -> Resultado:
        usuario = self.sistema.autenticar(self.sel_usuario.value, self.pin.value)
        self.pin.value = ""
        if usuario is None:
            return Resultado(False, "PIN incorreto.")
        self.usuario = usuario
        return Resultado(True, f"Bem-vindo(a), {usuario.nome}.")

    def _sair(self) -> Resultado:
        self.usuario = None
        return Resultado(True, "Sessão encerrada.")

    def _nivel3(self) -> Resultado:
        resultado = self.sistema.acionar_nivel3(self.usuario)
        if resultado.ok and self.evacuacao_auto.value:
            self.loc.evacuar(self.sistema.config.explosivo_xy, self.sistema.raio_m)
        return resultado

    def _mover(self, entrar: bool) -> Resultado:
        id_ = self.sel_pessoa.value
        centro, raio = self.sistema.config.explosivo_xy, self.sistema.raio_m
        if entrar:
            self.loc.entrar_no_raio(id_, centro, raio)
        else:
            self.loc.sair_do_raio(id_, centro, raio)
        return Resultado(True, f"{id_} caminhando para {'dentro' if entrar else 'fora'} do raio.")

    def _alternar_sinal(self) -> Resultado:
        t = self.loc.trabalhadores[self.sel_pessoa.value]
        self.loc.definir_sinal(t.id, not t.sinal)
        return Resultado(True, f"Localizador de {t.id} {'ligado' if t.sinal else 'desligado (simula bateria/sinal perdido)'}.")

    def _evacuar(self) -> Resultado:
        self.loc.evacuar(self.sistema.config.explosivo_xy, self.sistema.raio_m)
        return Resultado(True, "Todos caminhando para fora do raio.")

    # ------------------------------------------------------------ ciclo
    def _executar(self, acao: Optional[Callable[[], Optional[Resultado]]]) -> None:
        with self._lock:
            if acao is not None:
                try:
                    resultado = acao()
                except Exception as erro:
                    resultado = Resultado(False, f"Erro: {erro}")
                if resultado is not None:
                    cor = VERDE if resultado.ok else VERMELHO
                    self.mensagem.value = (f'<div style="font-family:Inter,Arial,sans-serif;padding:8px 12px;'
                                           f'border-radius:8px;background:{cor};color:#fff;font-weight:600">'
                                           f'{html.escape(resultado.mensagem)}</div>')
            self.sistema.atualizar()
            self.mapa.value = svg_mapa(self.sistema)
            self.status.value = html_status(self.sistema, self.usuario)
            self.registro.value = html_registro(self.sistema)

    def _laco(self) -> None:
        while not self._parar.wait(0.5):
            try:
                self._executar(None)
            except Exception:
                pass

    def mostrar(self) -> None:
        display(self.interface)
        threading.Thread(target=self._laco, daemon=True).start()

    def desligar(self) -> None:
        self._parar.set()
        self.audio.parar()

print("Sistema EXPRO carregado.")


In [ ]:
#@title 3. Painel do sistema { display-mode: "form" }
import importlib.util, subprocess, sys
if CONFIG["simulacao"]["usar_voz"] and importlib.util.find_spec("gtts") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gTTS"], check=False)

painel = PainelExpro(CONFIG)
painel.mostrar()


In [ ]:
#@title 4. (Opcional) Roteiro automático, sem cliques { display-mode: "form" }
demonstracao(CONFIG);
